In [ ]:
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resume.pdf")
documents = loader.load()

documents[:1]  # preview

c:\Users\KIIT0001\Desktop\Ap lab project\rag-resume\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-03-06T19:39:22+00:00', 'author': '', 'keywords': '', 'moddate': '2026-03-06T19:39:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Pankaj Kumar+91-9523984694\nBachelor of Technology (B.Tech) 2329196@kiit.ac.in\nComputer Science and Engineering GitHub\nKIIT UNIVERSITY ,BHUBANESW AR LinkedIn\nYear Degree/Certificate Institute CGPA/Percentage\n2023-2027 B.Tech Computer Science Engineering KIIT University, Bhubaneswar 8.15 CGPA\n2021 CBSE Class XII M.P.S 84.6%\n2019 CBSE Class X IHHS Academy 81.3%\nExperience\n•AICTE Virtual Internship in AI Machine Learning-Google AICTEJan 2025 - March 2025\nAll India Council for Technical Education Virtual\n–Completed hands-on training in Machine 

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
len(chunks)


8

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_6488\3055314890.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3976.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings # Updated 2026 import
import os

# 1. Your existing setup
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Build the vectorstore in RAM
vectorstore = FAISS.from_documents(chunks, embeddings)

# 3. ADD THIS LINE: Save the RAM data to a physical folder
# This will create a folder named 'faiss_index' containing your 2 missing files
vectorstore.save_local("faiss_index")

print("Successfully created 'faiss_index' folder with index.faiss and index.pkl")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7469.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully created 'faiss_index' folder with index.faiss and index.pkl


In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama3-70b-8192",
    temperature=0
)


In [12]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


response = rag_chain.invoke("what is my qualifications?")
print(response)

Based on the provided context, your qualifications include:

1. Key courses taken:
   - Core Computer Science: Data Structures and Algorithms, Object-Oriented Programming, Operating Systems, Database Management Systems, Computer Networks
   - Programming: C Programming, Java Programming, Python Programming

2. Certifications:
   - Red Hat Certified System Administrator (RHCSA) – Red Hat
   - Certified in Linux system administration

3. Hands-on experience:
   - Linux system administration
   - Shell scripting
   - System configuration

4. Comprehensive training:
   - Alpha DSA with Java – Apna College (completed training in data structures, algorithms, and problem solving using Java)

5. Positions of Responsibility:
   - Finance and Marketing Team Member, KRAYA & KUBER Society, KIIT (2023)
